# Evaluating LLM Outputs using LLUMO API

#### This notebook evaluates the quality of LLM-generated responses based on key metrics such as confidence, relevance, and accuracy etc. The evaluation is performed using the LLUMO API.

# Setting up the environment

# Importing libraries:

* requests: A popular library for making HTTP requests in Python.</br>
* json: Used for parsing JSON data, which is common in API responses.</br>
* from google.colab : Used to securely store and retrieve user-specific data in Google Colab.</br>
* pandas: A powerful library for data manipulation and analysis, providing data structures like DataFrames and Series. It is commonly used for cleaning, transforming, and analyzing structured data (e.g., CSV files, Excel spreadsheets, SQL databases)

In [1]:
import requests
import json
import time
import pandas as pd
from google.colab import userdata

### Loading the dataset

We begin by loading a dataset containing:

Knowledge Base: The input question or query.</br>
User Question: The context retrieved for the query.</br>
LLM Responses: The expected correct response.</br>


In [6]:
# You can get your own sample data by visiting https://app.llumo.ai/evallm In sample data you will find the dataset to play and experiment
df=pd.read_excel('/content/Sample_QA.xlsx').head(5)

In [8]:
df

,Knowledge Base,User Question,Prompt
0,StellarSolutions is your trusted partner in in...,Can I get a refund if the CRM platform doesn't...,"Yes, StellarSolutions offers a 30-day money-ba..."
1,DigitalDepot is your one-stop shop for the lat...,How can I track my order from DigitalDepot?,The provided text doesn't specify an order tra...
2,ProMark Solutions is your trusted partner in m...,How can ProMark Solutions help my business inc...,ProMark Solutions enhances brand visibility th...
3,Skyline Logistics is your reliable partner for...,How can I track my shipment with Skyline Logis...,Skyline Logistics' website or customer service...
4,BuildMaster Construction is your expert partne...,What are the key phases of a construction proj...,BuildMaster Construction's project phases enco...


##  Define API Configuration


In [9]:
# Define the endpoint, headers, and payload

LLUMOAI_KEY = userdata.get("LLUMOAI_API_KEY")  # Ensure this is set in the environment(Visit https://app.llumo.ai/ to get your own Api key)
if not LLUMOAI_KEY:
    raise ValueError("Missing OpenAI API key. Set OPENAI_API_KEY as an environment variable.")


LLUMO_ENDPOINT = "https://app.llumo.ai/api/create-eval-analytics"
headers = {
    "Authorization": f"Bearer {LLUMOAI_KEY}",
    "Content-Type": "application/json"
}

## Define Prompt

In [10]:
Prompt="""
You are a highly intelligent AI assistant designed to provide accurate, concise, and helpful responses based on the knowledge available in the provided database. Your primary goal is to ensure that the user receives the most relevant information in a clear and straightforward manner. If the knowledge base does not contain enough data to fully address the user's query, you should politely inform them of the limitation. Additionally, whenever possible, suggest alternative sources or methods for obtaining the information they seek. Your responses should be user-friendly, approachable, and precise, ensuring the user leaves the interaction satisfied.

*Knowledge Base:*
{{Knowledge Base}}

*User Question:*
{{User Question}}

*Your response:*
"""

# Process each query and Evaluate

In [13]:
for indx, row in df.iterrows():
    Knowledge_base = row["Knowledge Base"]
    User_Question = row["User Question"]
    llm_output=row['Prompt']
    # Construct request payload for API
    req_body = {
        "prompt":Prompt,
        "input": {
            "Knowledge Base": Knowledge_base,
            "User Question": User_Question,
        },
        "output": llm_output,
        "analytics": ["confidence", "relevance", "accuracy"]  # Specify evaluation metrics
    }

    try:
        # Send POST request to LLUMO API
        response = requests.post(url=LLUMO_ENDPOINT, json=req_body, headers=headers, timeout=18)
        response_json = response.json()

        # Extract data from response
        response_data = json.loads(response_json["data"]["data"])
        print(response_data)  # Debugging print statement

        print(f"✅ Processed row {indx+1}")

    except Exception as e:
        print(f"❌ Error at row {indx+1}: {e}")
        continue


{'analyticsScore': {'confidence': 75, 'relevance': 90, 'accuracy': 100, 'context': 80, 'clarity': 95, 'overallScore': 88}, 'reasoning': {'confidence': ['The response is concise and directly answers the question.', 'The language used is clear and straightforward, conveying confidence.', 'There are no hedging phrases or expressions of uncertainty.', 'However,  a slightly more detailed response, perhaps mentioning the conditions of the guarantee, would enhance confidence further.'], 'relevance': ["The response is highly relevant to the user's question about refunds.", 'It directly addresses the core element of the query: the availability of a money-back guarantee.', 'The information provided is precisely what the user needs to know.'], 'accuracy': ['The response accurately reflects the information provided in the knowledge base.', 'The mention of the 30-day money-back guarantee is correct and precise.', 'There are no inaccuracies or misrepresentations of the facts.'], 'context': ["The res

# Conclusion
### Throughout this notebook, we have explored the process of evaluation a chatbot for a QA-RAG application using Llumo Api, Python. We covered essential steps, including setting up the environment, loading and preparing cdata, extracting relevant context, and evaluating the llm outputs using Llumo.